# EDS object detection: YOLO11n, YOLO11s, and DETR

**Status (6 November 2026): the manifest-backed Roboflow v4 reviewed export has completed three training and validation runs and one selected-model final test.** See the recorded results below and the [archived report](../Obj-Dect-OCR/object_detection/01760a841333/runs/v4-reviewed-01/RESULTS.md). The earlier baseline remains a separate experiment.

**V9 baseline (26 November 2026): YOLO11n and YOLO11s training logs were supplied; DETR has no measured v9 result.** The raw `data/training/Front EDS.v9i.coco` export remains untouched. The separate `data/training/2026-11-25-roboflow-v9-stem-isolated` copy uses a conservative filename-stem rule, excludes 427 overlapping/repeated images including three visually confirmed train/test scene variants, and has a verified checksum manifest plus `split_isolation_audit.json`. Its 6,858/294/299 train/valid/test images still need human source and annotation review. Human source and annotation review remains open. The v9 YOLO figures below are measured; the earlier three-model benchmark uses a different dataset.

This notebook implements the object detection allocation: dataset/label audit, pretrained model selection, fine-tuning, validation, per-class mAP, speed/failure comparison, threshold selection, and a final held-out test. It retains train/valid/test separation and records settings and hashes. Run from the repo's `notebooks/` folder or repo root.

Sources: [task allocation PDF](file:///C:/Users/brend/Downloads/CVDL_Task_Allocation.pdf), [training inventory](../data/training/README.md), [readiness handoff](../docs/data_preparation/TRAINING_READINESS.md), [annotation guide](../docs/annotation/guide.md), [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11), [Hugging Face DETR](https://huggingface.co/docs/transformers/v4.51.1/en/model_doc/detr).

## Environment and run controls

Use a dedicated Python environment. Install a PyTorch build appropriate to this machine, then `ultralytics==8.3.158 transformers>=4.52,<5 timm torchmetrics pycocotools pyyaml pillow nbformat ipykernel`. Keep it separate from the pinned data-preparation environment, which uses Transformers 5.x. Pretrained weights may download when training is enabled. The first run is a safe preflight. Train each model separately with `scripts/train_object_detection.py`.

In [1]:
"""Run this notebook cell first in a fresh Google Colab GPU runtime."""

import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Local kernel: Colab setup skipped; use the existing local paths.")
else:
    drive.mount("/content/drive")
    drive_folder = Path("/content/drive/MyDrive/CVDL-P1-14-colab")
    code_zip = drive_folder / "CVDL-P1-14-v9-colab-code.zip"
    data_zip = drive_folder / "2026-11-25-roboflow-v9-stem-isolated.zip"
    for archive in (code_zip, data_zip):
        if not archive.is_file():
            raise FileNotFoundError(f"Upload this ZIP to Google Drive first: {archive}")

    def extract_checked(archive, destination):
        destination.mkdir(parents=True, exist_ok=True)
        base = destination.resolve()
        with zipfile.ZipFile(archive) as bundle:
            for member in bundle.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base):
                    raise ValueError(f"Unsafe ZIP entry: {member.filename}")
            bundle.extractall(base)

    repo_root = Path("/content/CVDL-P1-14")
    extract_checked(code_zip, Path("/content"))
    if not (repo_root / "src").is_dir():
        raise RuntimeError("The code ZIP did not create /content/CVDL-P1-14/src")

    dataset_root = repo_root / "data/training/2026-11-25-roboflow-v9-stem-isolated"
    if not (dataset_root / "preparation_manifest.json").is_file():
        local_zip = Path("/content/2026-11-25-roboflow-v9-stem-isolated.zip")
        shutil.copy2(data_zip, local_zip)
        try:
            extract_checked(local_zip, repo_root / "data/training")
        finally:
            local_zip.unlink(missing_ok=True)
    if not (dataset_root / "preparation_manifest.json").is_file():
        raise RuntimeError(f"The dataset ZIP did not create {dataset_root}")

    results_root = drive_folder / "results"
    output_root = results_root / ".runtime/object_detection_benchmark"
    archive_root = results_root / "Obj-Dect-OCR/object_detection"
    for folder in (output_root, archive_root):
        folder.mkdir(parents=True, exist_ok=True)
    os.environ.update({
        "EDS_REPO_ROOT": str(repo_root),
        "EDS_DATASET_ROOT": str(dataset_root),
        "EDS_OUTPUT_ROOT": str(output_root),
        "EDS_ARCHIVE_ROOT": str(archive_root),
        "EDS_YOLO_VIEW_ROOT": str(repo_root / ".runtime/object_detection_benchmark"),
        "EDS_PRETRAINED_ROOT": str(repo_root / ".runtime/object_detection_benchmark/pretrained"),
        "YOLO_CONFIG_DIR": str(repo_root / ".runtime/object_detection_benchmark/ultralytics"),
        "USE_TF": "0",
    })
    os.chdir(repo_root)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.158",
        "transformers>=4.52,<5", "timm", "torchmetrics", "pycocotools",
        "pyyaml", "pillow", "nbformat", "ipykernel",
    ], check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Select Kernel > Colab > New Colab Server and choose a GPU machine."
        )
    print({"repo": str(repo_root), "dataset": str(dataset_root),
           "outputs": str(output_root), "archive": str(archive_root),
           "gpu": torch.cuda.get_device_name(0)})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'repo': '/content/CVDL-P1-14', 'dataset': '/content/CVDL-P1-14/data/training/2026-11-25-roboflow-v9-stem-isolated', 'outputs': '/content/drive/MyDrive/CVDL-P1-14-colab/results/.runtime/object_detection_benchmark', 'archive': '/content/drive/MyDrive/CVDL-P1-14-colab/results/Obj-Dect-OCR/object_detection', 'gpu': 'Tesla T4'}


In [2]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Wed Nov 25 18:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |    1155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, sys, time
import re
from collections import Counter

ROOT = Path(os.environ.get('EDS_REPO_ROOT', Path.cwd())).resolve()
if not (ROOT / 'src').is_dir():
    ROOT = next((p for p in Path.cwd().resolve().parents if (p / 'src').is_dir()), ROOT)
if not (ROOT / 'src').is_dir():
    raise RuntimeError(f'Repo not found from {Path.cwd()}. In Colab, run the Colab setup cell first.')
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
from object_detection_artifacts import archive_benchmark

DATASET_ROOT = Path(os.environ.get('EDS_DATASET_ROOT', ROOT / 'data/training/2026-11-25-roboflow-v9-stem-isolated')).resolve()
OUTPUT_ROOT = Path(os.environ.get('EDS_OUTPUT_ROOT', ROOT / '.runtime/object_detection_benchmark')).resolve()
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / '.runtime/object_detection_benchmark/ultralytics'))
os.environ.setdefault('USE_TF', '0')  # Keep this detector kernel on the PyTorch backend.
os.environ.setdefault('HF_HOME', str(OUTPUT_ROOT / 'huggingface'))
RUN_PREPARE_YOLO = False  # Optional dataset conversion; training CLI prepares it too.
RUN_TRAINING = False    # Enable manually after reviewing the v9 split audit and verified manifest.
TRAIN_MODEL = 'yolo11n'   # Choose: 'yolo11n', 'yolo11s', or 'detr-resnet-50'.
RUN_BENCHMARK = False     # Enable for completed v9 models selected below.
RUN_FINAL_TEST = False    # Keep held-out v9 test closed until validation-based selection.
RUN_ID = 'v9-2026-11-25-01'  # New comparison set; keep this ID for all three models.
BENCHMARK_MODELS = ('yolo11n', 'yolo11s')  # Add DETR only after its v9 run completes.
YOLO_EPOCHS, DETR_EPOCHS, IMAGE_SIZE, BATCH_SIZE, SEED = 60, 60, 640, 32, 42
OPERATING_CONF, YOLO_NMS_IOU = 0.30, 0.70  # Starting diagnostic threshold; select v9 settings on validation.
DEVICE = 'cuda' if importlib.util.find_spec('torch') and __import__('torch').cuda.is_available() else 'cpu'
CLASS_NAMES = ('front_eds', 'front_eds_num', 'front_eds_route', 'front_eds_destination')
COCO_TO_INDEX = {1: 0, 3: 1, 4: 2, 2: 3}  # Explicit IDs from the readiness handoff.
INDEX_TO_NAME = dict(enumerate(CLASS_NAMES))
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', RUN_ID):
    raise ValueError('RUN_ID may contain only letters, digits, underscores and hyphens.')
if TRAIN_MODEL not in ('yolo11n', 'yolo11s', 'detr-resnet-50'):
    raise ValueError(f'Unsupported TRAIN_MODEL: {TRAIN_MODEL!r}')
if RUN_TRAINING and (RUN_BENCHMARK or RUN_FINAL_TEST):
    raise ValueError('Train and benchmark/test in separate Run All executions.')
if RUN_TRAINING and DEVICE != 'cuda':
    raise RuntimeError('Select a CUDA GPU kernel before starting training.')
print({'repo': str(ROOT), 'dataset': str(DATASET_ROOT), 'device': DEVICE,
       'run_id': RUN_ID, 'train_model': TRAIN_MODEL, 'training': RUN_TRAINING,
       'prepare_yolo': RUN_PREPARE_YOLO, 'benchmark': RUN_BENCHMARK, 'final_test': RUN_FINAL_TEST})

{'repo': '/content/CVDL-P1-14', 'dataset': '/content/CVDL-P1-14/data/training/2026-11-25-roboflow-v9-stem-isolated', 'device': 'cuda', 'run_id': 'v9-2026-11-25-01', 'train_model': 'detr-resnet-50', 'training': False, 'prepare_yolo': False, 'benchmark': True, 'final_test': False}


## 1. Data and class audit

The four foreground classes are **whole front/side panel**, **service number**, **intermediate route or landmark text**, and **final destination**. `front_eds` is the historical name for both front and side panels. The unused COCO category `0: front-eds` is not a fifth foreground class. The guide distinguishes route from destination by meaning and typography, not by whether text appears above or below another line. A sign with several labelled parts can contain several boxes, including a containing whole-panel box.

An image with zero **reviewed** bounding boxes is a negative example and remains in every relevant split with an empty target. An image merely missing annotations must be reviewed before being declared negative. Rear signs and unrelated bus-stop text are negatives under the current scope.

**Integration gap to resolve:** the current reader adapter filters model class IDs `[1, 2, 3]`, and its association service recognises labels `eds`, `bus_number`, and `destination`. A new four-class checkpoint trained with the indices below would emit `front_eds` at index 0 and `front_eds_route` at index 2. Update the adapter's ID/name mapping and the association/output contract before deploying it; otherwise the whole-panel class can be dropped and route boxes can be ignored. Keep the training taxonomy stable rather than renaming labels inside the training data to match legacy app labels.

The uploaded `2026-10-24` Roboflow export has 420/39/20 images but no manifest. The separately built `2026-10-30-torchvision` export contains 411/39/20 images with a manifest; this notebook uses the latter by default. These are distinct dataset versions and their results must not be mixed. Do not overwrite the Roboflow folder or claim it was produced by the torchvision recipe.

In [4]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

DATA_READY, splits, inventory, data_error = False, None, {}, None
try:
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f'Full training export is absent: {DATASET_ROOT}')
    from data_preparation.manifests import verify_export
    from data_preparation.datasets import load_coco_splits
    manifest = verify_export(DATASET_ROOT)
    if manifest is None:
        raise ValueError('No verified export manifest. Use a new manifest-backed version; do not synthesize one from counts.')
    splits = load_coco_splits(DATASET_ROOT)
    for split_name, split in splits.items():
        categories = {c['id']: c['name'] for c in split['data']['categories']}
        for coco_id, index in COCO_TO_INDEX.items():
            if categories.get(coco_id) != CLASS_NAMES[index]:
                raise ValueError(f'{split_name}: COCO category {coco_id} is {categories.get(coco_id)!r}, expected {CLASS_NAMES[index]!r}')
        used = {a['category_id'] for a in split['data']['annotations']}
        if not used <= set(COCO_TO_INDEX):
            raise ValueError(f'{split_name}: unexpected labelled category IDs {used - set(COCO_TO_INDEX)}')
        inventory[split_name] = {'images': len(split['rows']),
                                 'negative_images': sum(not r['annotations'] for r in split['rows']),
                                 'boxes_by_class': {CLASS_NAMES[i]: sum(a['category_id'] == coco_id for a in split['data']['annotations'])
                                                    for coco_id, i in COCO_TO_INDEX.items()},
                                 'coco_sha256': sha256(split['folder'] / '_annotations.coco.json')}
    manifest_path = next(p for p in (DATASET_ROOT / 'augmentation_manifest.json', DATASET_ROOT / 'preparation_manifest.json') if p.is_file())
    DATA_FINGERPRINT = sha256(manifest_path)
    DATA_READY = True
except (FileNotFoundError, ValueError, ImportError) as exc:
    data_error = str(exc)

print(json.dumps({'data_ready': DATA_READY, 'reason_if_not_ready': data_error,
                  'inventory': inventory}, indent=2))

{
  "data_ready": true,
  "reason_if_not_ready": null,
  "inventory": {
    "train": {
      "images": 6858,
      "negative_images": 579,
      "boxes_by_class": {
        "front_eds": 7827,
        "front_eds_num": 7410,
        "front_eds_route": 5133,
        "front_eds_destination": 5505
      },
      "coco_sha256": "5594cb4b2d1cd7f2754f6f4281441164f25da71870038b2cdc522a6b447c6922"
    },
    "valid": {
      "images": 294,
      "negative_images": 20,
      "boxes_by_class": {
        "front_eds": 348,
        "front_eds_num": 335,
        "front_eds_route": 230,
        "front_eds_destination": 241
      },
      "coco_sha256": "0fb320ea1460090ecb3e0252306da1dc093d96a92b2fc5c6f1d53e520d152b2c"
    },
    "test": {
      "images": 299,
      "negative_images": 20,
      "boxes_by_class": {
        "front_eds": 354,
        "front_eds_num": 340,
        "front_eds_route": 239,
        "front_eds_destination": 232
      },
      "coco_sha256": "dd74d8c5c031886e7385dd45f87dc6229265

### Existing measured result (historical context only)

The branch includes one prior YOLO11n field-detector report. Its 13-image deduplicated subset evaluated only service-number and destination classes from a different bundled YOLO dataset, on a Mac CPU. It is not the four-class EDS benchmark below. The next cell reads the recorded values and limitations directly, rather than transcribing them as new measurements.

In [5]:
historical_path = ROOT / 'Obj-Dect-OCR/reports/detection.json'
if historical_path.is_file():
    historical = json.loads(historical_path.read_text(encoding='utf-8'))
    print(json.dumps({'run_id': historical['run_id'], 'images': historical['dataset']['image_count'],
                      'classes': historical['detector']['classes'],
                      'metrics': historical['metrics'], 'hardware': historical['hardware'],
                      'limitations': historical['limitations']}, indent=2))
else:
    print('No historical detector report in this checkout.')

{
  "run_id": "field-detector-20261022T141403Z",
  "images": 13,
  "classes": [
    1,
    2
  ],
  "metrics": {
    "precision": 0.9614429997637921,
    "recall": 1.0,
    "map_50": 0.995,
    "map_50_95": 0.7486922272522737,
    "whole_eds_map": null,
    "speed_ms_per_image": {
      "preprocess": 0.24676284654280886,
      "inference": 26.947298073747124,
      "loss": 0.0007756909606261895,
      "postprocess": 0.2769359181491801
    }
  },
  "hardware": {
    "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
    "machine": "arm64",
    "processor": "arm",
    "torch_threads": 4,
    "torch": "2.9.0",
    "ultralytics": "8.3.158"
  },
  "limitations": [
    "Only service-number and destination classes [1,2] are evaluated; no whole-EDS accuracy claim.",
    "Exact training/test duplicates and unlabelled images excluded; near-duplicate and original training provenance are unverified.",
    "Small bundled image subset; does not establish performance on unseen video or other EDS form

**Audit finding:** automated checks pass for the labelled source and the new export's manifest, images, boxes and class IDs. Inspect per-class box counts and negative counts in the output above. Do not treat `-1` AP (no ground truth for a class) as zero. Capture/recording grouping and semantic review of small, distant signs still require human evidence. The 420-image Roboflow snapshot is not certified by this notebook.

## 2. Convert only the verified export for YOLO; keep original COCO for DETR

The conversion copies images into ignored `.runtime/`, writes one YOLO label file per image (empty for a reviewed negative), and creates an explicit class-name YAML. It does not mutate source COCO files. Conversion is blocked while `DATA_READY` is false.

In [6]:
def make_yolo_view():
    import shutil, yaml
    from PIL import Image
    view = Path(os.environ.get('EDS_YOLO_VIEW_ROOT', OUTPUT_ROOT)).resolve() / DATA_FINGERPRINT[:12] / 'yolo'
    if view.exists():
        yaml_path = view / 'data.yaml'
        if not yaml_path.is_file():
            raise FileExistsError(f'Incomplete existing conversion: {view}. Inspect it before rerunning.')
        print('Reusing existing conversion:', yaml_path)
        return yaml_path
    for split_name, split in splits.items():
        image_dir, label_dir = view / 'images' / split_name, view / 'labels' / split_name
        image_dir.mkdir(parents=True)
        label_dir.mkdir(parents=True)
        for row in split['rows']:
            src = row['path']
            with Image.open(src) as opened:
                format_name = opened.format
                needs_png = format_name == 'AVIF' or src.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
                dest_name = src.stem + '.png' if needs_png else src.name
                if (image_dir / dest_name).exists() or (label_dir / (src.stem + '.txt')).exists():
                    raise ValueError(f'Duplicate output stem in {split_name}: {src.stem}')
                if needs_png:
                    opened.convert('RGB').save(image_dir / dest_name, format='PNG')
                else:
                    shutil.copy2(src, image_dir / dest_name)
            width, height = row['image']['width'], row['image']['height']
            lines = []
            for annotation in row['annotations']:
                x, y, w, h = annotation['bbox']
                index = COCO_TO_INDEX[annotation['category_id']]
                lines.append(f'{index} {(x + w/2)/width:.8f} {(y + h/2)/height:.8f} {w/width:.8f} {h/height:.8f}')
            (label_dir / (src.stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
    yaml_path = view / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({'path': str(view), 'train': 'images/train', 'val': 'images/valid',
                                          'test': 'images/test', 'names': INDEX_TO_NAME}, sort_keys=False), encoding='utf-8')
    return yaml_path

YOLO_DATA_YAML = None
if DATA_READY and RUN_PREPARE_YOLO:
    YOLO_DATA_YAML = make_yolo_view()
    print('Converted verified COCO export:', YOLO_DATA_YAML)
else:
    print('YOLO conversion skipped; training CLI can prepare it when needed.')

YOLO conversion skipped; training CLI can prepare it when needed.


## Manual model training

Select one candidate in the first code cell with `TRAIN_MODEL` and set `RUN_TRAINING=True`. Keep `RUN_BENCHMARK=False` and `RUN_FINAL_TEST=False` during training, then click **Run All**. Use the same `RUN_ID` for YOLO11n, YOLO11s and DETR when they belong to one comparison, such as `baseline-01`; each model still gets its own folder. Change the ID only when starting another experiment set. Training starts from the published pretrained weights and refuses to overwrite the same model under an existing run ID. It does not resume an earlier EDS checkpoint.

After training finishes, set `RUN_TRAINING=False`, keep the same `RUN_ID`, select the model in `BENCHMARK_MODELS`, and enable `RUN_BENCHMARK` in a separate Run All execution.

In [7]:
if DATA_READY and RUN_TRAINING:
    import subprocess
    training_epochs = YOLO_EPOCHS if TRAIN_MODEL.startswith('yolo') else DETR_EPOCHS
    command = [sys.executable, str(ROOT / 'scripts/train_object_detection.py'),
               '--model', TRAIN_MODEL, '--run-id', RUN_ID,
               '--dataset', str(DATASET_ROOT), '--epochs', str(training_epochs),
               '--batch-size', str(BATCH_SIZE), '--image-size', str(IMAGE_SIZE),
               '--seed', str(SEED)]
    print('Starting fresh model training:', ' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Training paused. Set RUN_TRAINING=True when the model and run ID are ready.')

Training paused. Set RUN_TRAINING=True when the model and run ID are ready.


### V9 baseline and YOLO11s tuning

Measured from the supplied v9 `results.csv` files (best validation epoch):

| Model | Best epoch | Validation mAP50 | Validation mAP50:95 |
| --- | ---: | ---: | ---: |
| YOLO11n | 8 | 0.92966 | 0.67547 |
| YOLO11s | 9 | 0.93337 | **0.69106** |

The YOLO11s log records a resumed run. DETR has no measured v9 result, so this is a two-model v9 comparison. No tuning or v9 final-test scores are claimed here.

Two unrun YOLO11s variants test specific hypotheses: `gentle_aug` adds small shifts, zoom and brightness variation to address overfitting; `localization` uses 800-pixel input and a slightly higher box-loss gain to test tighter text boxes. Both start from published pretrained weights, use the same reviewed v9 split and seed, and save to new folders. Compare best validation mAP50:95, per-class AP and negative-image false positives before opening test.

Run the code cell below **once per variant** after the data audit and YOLO conversion cell. Set `RUN_TUNING=True` and change `TUNING_VARIANT`. Keep the baseline training, benchmark and final-test switches off. Existing checkpoints are not overwritten.

In [ ]:
RUN_TUNING = False  # Set True only after reviewing the v9 data audit.
TUNING_VARIANT = "gentle_aug"  # Then run again with "localization".

if RUN_TUNING:
    if not DATA_READY:
        raise RuntimeError("The dataset audit must pass before tuning.")
    if RUN_TRAINING or RUN_BENCHMARK or RUN_FINAL_TEST:
        raise RuntimeError("Run tuning separately from baseline training and evaluation.")
    from ultralytics import YOLO

    variants = {
        "gentle_aug": {"imgsz": 640, "batch": 32,
                       "translate": 0.02, "scale": 0.10, "hsv_v": 0.08},
        "localization": {"imgsz": 800, "batch": 16, "box": 8.5},
    }
    if TUNING_VARIANT not in variants:
        raise ValueError(f"Unknown tuning variant: {TUNING_VARIANT}")

    project = OUTPUT_ROOT / DATA_FINGERPRINT[:12] / "runs" / "v9-tuning-01"
    name = f"yolo11s-{TUNING_VARIANT}"
    if (project / name).exists():
        raise FileExistsError(f"Existing run will not be overwritten: {project / name}")
    settings = {
        "data": str(make_yolo_view()), "epochs": 60, "patience": 15,
        "seed": 42, "deterministic": True, "device": 0, "workers": 0,
        "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0,
        "degrees": 0.0, "translate": 0.0, "scale": 0.0, "fliplr": 0.0,
        "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
    }
    settings.update(variants[TUNING_VARIANT])
    YOLO("yolo11s.pt").train(
        **settings, project=str(project), name=name, exist_ok=False,
    )
    print("Tuning output:", project / name)
else:
    print("YOLO11s tuning paused. Set RUN_TUNING=True to run one variant.")

## 4. Common evaluation: all four classes, all splits, and negatives

Measure box mAP@0.50 and mAP@0.50:0.95 per class using the same COCO targets and TorchMetrics implementation for each selected candidate. Include zero-box images: predictions on them are false positives. Also report false positives per negative image at score ≥ 0.30, median/p95 end-to-end detector inference milliseconds per image, and sample count. Train metrics describe fit and are optimistic; validation is for selection; test is held out. `RUN_BENCHMARK=True` evaluates train+valid for `BENCHMARK_MODELS`, even if only one checkpoint exists. Each selection writes a separate JSON report. `RUN_FINAL_TEST=True` evaluates test after validation-based model and thresholds are frozen. The small historical YOLO report in `Obj-Dect-OCR/reports/detection.json` is separate: it covers two field classes on 13 images from another dataset and cannot fill this table.

In [8]:
MODEL_PATHS, TRAINING_HISTORY = {}, {}
if DATA_READY:
    saved_runs = OUTPUT_ROOT / DATA_FINGERPRINT[:12] / 'runs' / RUN_ID
    for candidate in ('yolo11n', 'yolo11s'):
        run_dir = saved_runs / candidate
        checkpoint = run_dir / 'weights/best.pt'
        if checkpoint.is_file() and (run_dir / 'complete.txt').is_file():
            MODEL_PATHS[candidate] = str(checkpoint)
            TRAINING_HISTORY[candidate] = {'history_csv': str(run_dir / 'results.csv')}
    detr_dir = saved_runs / 'detr-resnet-50'
    if (detr_dir / 'config.json').is_file() and (detr_dir / 'complete.txt').is_file():
        MODEL_PATHS['detr-resnet-50'] = str(detr_dir)
print('Completed runs for evaluation:', MODEL_PATHS)

Completed runs for evaluation: {'yolo11n': '/content/drive/MyDrive/CVDL-P1-14-colab/results/.runtime/object_detection_benchmark/43159c679cbe/runs/v9-2026-11-25-01/yolo11n/weights/best.pt', 'yolo11s': '/content/drive/MyDrive/CVDL-P1-14-colab/results/.runtime/object_detection_benchmark/43159c679cbe/runs/v9-2026-11-25-01/yolo11s/weights/best.pt'}


In [ ]:
BENCHMARK = {}
EXPECTED_MODELS = {'yolo11n', 'yolo11s', 'detr-resnet-50'}
if RUN_BENCHMARK:
    if not BENCHMARK_MODELS or len(set(BENCHMARK_MODELS)) != len(BENCHMARK_MODELS):
        raise ValueError('BENCHMARK_MODELS must list each selected model once.')
    missing = set(BENCHMARK_MODELS) - set(MODEL_PATHS)
    if missing:
        raise FileNotFoundError(f'Complete checkpoint required for: {sorted(missing)}')
if DATA_READY and RUN_BENCHMARK:
    import numpy as np
    import torch
    from PIL import Image
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    from transformers import DetrForObjectDetection, DetrImageProcessor
    from ultralytics import YOLO

    def target_for(row):
        boxes = [[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]]
                 for a in row['annotations']]
        labels = [COCO_TO_INDEX[a['category_id']] for a in row['annotations']]
        return {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                'labels': torch.tensor(labels, dtype=torch.int64)}

    def make_predictor(name, location):
        if name.startswith('yolo'):
            model = YOLO(location)
            def predict(image):
                bgr = np.asarray(image)[:, :, ::-1].copy()  # Ultralytics NumPy input expects OpenCV BGR.
                out = model.predict(bgr, imgsz=IMAGE_SIZE, conf=0.001,
                                    iou=YOLO_NMS_IOU, device=DEVICE, verbose=False)[0]
                b = out.boxes
                return {'boxes': b.xyxy.cpu().float(), 'scores': b.conf.cpu().float(),
                        'labels': b.cls.cpu().long()}
            return predict
        proc = DetrImageProcessor.from_pretrained(location)
        model = DetrForObjectDetection.from_pretrained(location).to(DEVICE).eval()
        def predict(image):
            inputs = proc(images=image, size={'shortest_edge': IMAGE_SIZE, 'longest_edge': IMAGE_SIZE},
                          return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.inference_mode():
                out = model(**inputs)
            pred = proc.post_process_object_detection(out, threshold=0.001,
                target_sizes=torch.tensor([[image.height, image.width]], device=DEVICE))[0]
            return {k: v.cpu() for k, v in pred.items() if k in ('boxes', 'scores', 'labels')}
        return predict

    def evaluate_split(predict, rows):
        metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
        metric50 = MeanAveragePrecision(box_format='xyxy', iou_thresholds=[0.5], class_metrics=True)
        latencies, negative_fp, negatives = [], 0, 0
        if rows:
            with Image.open(rows[0]['path']) as opened:
                warm_image = opened.convert('RGB')
            for _ in range(3): predict(warm_image)  # Keep cold model/cache startup out of timing.
        for row in rows:
            with Image.open(row['path']) as opened:
                image = opened.convert('RGB')
            if DEVICE == 'cuda': torch.cuda.synchronize()
            started = time.perf_counter()
            pred = predict(image)
            if DEVICE == 'cuda': torch.cuda.synchronize()
            latencies.append(1000 * (time.perf_counter() - started))
            target = target_for(row)
            metric.update([pred], [target])
            metric50.update([pred], [target])
            if not row['annotations']:
                negatives += 1
                negative_fp += int((pred['scores'] >= OPERATING_CONF).sum())
        result, result50 = metric.compute(), metric50.compute()
        by_class = {}
        for i, class_id in enumerate(result['classes'].tolist()):
            j = result50['classes'].tolist().index(class_id)
            ap50, ap = float(result50['map_per_class'][j]), float(result['map_per_class'][i])
            by_class[CLASS_NAMES[class_id]] = {
                'map50': None if ap50 < 0 else ap50,
                'map50_95': None if ap < 0 else ap,
                'mar_100': float(result['mar_100_per_class'][i])}
        return {'images': len(rows), 'negatives': negatives,
                'map50': float(result50['map_50']), 'map50_95': float(result['map']),
                'mar_100': float(result['mar_100']),
                'by_class': by_class, 'false_positives_on_negatives_at_operating_conf': negative_fp,
                'median_inference_ms': float(np.median(latencies)),
                'p95_inference_ms': float(np.percentile(latencies, 95))}

    for name in BENCHMARK_MODELS:
        location = MODEL_PATHS[name]
        predict = make_predictor(name, location)
        BENCHMARK[name] = {'train': evaluate_split(predict, splits['train']['rows']),
                           'valid': evaluate_split(predict, splits['valid']['rows'])}
        if RUN_FINAL_TEST:
            BENCHMARK[name]['test'] = evaluate_split(predict, splits['test']['rows'])
        print(name, json.dumps(BENCHMARK[name], indent=2))
else:
    print('Benchmark paused. Enable RUN_BENCHMARK and choose completed BENCHMARK_MODELS.')

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


## 5. Validation threshold and overlap suppression

mAP sweeps confidence thresholds internally, so evaluate it with a low prediction cutoff (`0.001`). Separately choose the **operating** confidence threshold using validation only. The following code counts class-correct matches at IoU ≥ 0.50 (one prediction per ground-truth box), along with false positives on reviewed negative images. It reports a threshold sweep for the highest-validation-mAP candidate. Inspect the precision/recall tradeoff and OCR crops before freezing the setting. For YOLO, repeat the validation sweep with NMS IoU 0.50 and 0.70 if duplicate boxes are a problem; DETR does not require NMS by design.

In [ ]:
THRESHOLD_SWEEP = []
if DATA_READY and BENCHMARK:
    import torch
    from PIL import Image
    from torchvision.ops import box_iou

    selected = max(BENCHMARK, key=lambda name: BENCHMARK[name]['valid']['map50_95'])
    predict = make_predictor(selected, MODEL_PATHS[selected])
    cached = []
    for row in splits['valid']['rows']:
        with Image.open(row['path']) as opened:
            pred = predict(opened.convert('RGB'))
        cached.append((pred, target_for(row)))

    for threshold in (0.10, 0.20, 0.30, 0.50, 0.70):
        tp = fp = fn = negative_fp = 0
        for pred, target in cached:
            keep = (pred['scores'] >= threshold).nonzero(as_tuple=True)[0]
            keep = keep[torch.argsort(pred['scores'][keep], descending=True)]
            assigned = set()
            for index in keep.tolist():
                same = (target['labels'] == pred['labels'][index]).nonzero(as_tuple=True)[0]
                same = [j for j in same.tolist() if j not in assigned]
                if same:
                    ious = box_iou(pred['boxes'][index:index+1], target['boxes'][same])[0]
                    best = int(torch.argmax(ious))
                    if float(ious[best]) >= 0.50:
                        tp += 1
                        assigned.add(same[best])
                        continue
                fp += 1
                if not len(target['labels']): negative_fp += 1
            fn += len(target['labels']) - len(assigned)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        THRESHOLD_SWEEP.append({'model': selected, 'threshold': threshold,
                                'precision_iou50': precision, 'recall_iou50': recall,
                                'f1_iou50': 2*precision*recall/(precision+recall) if precision+recall else 0.0,
                                'negative_false_positives': negative_fp})
    print(json.dumps(THRESHOLD_SWEEP, indent=2))
else:
    print('Threshold selection pending validation predictions.')

### Recorded `v4-reviewed-01` benchmark and final test

The reviewed v4 export has 735 train, 53 validation and 53 test images. All three models trained for up to 60 epochs with seed 42, 640-pixel input and CUDA. The validation comparison selected YOLO11s by mAP50:95; only it received a final held-out test.

| Model | Validation mAP50 / mAP50:95 | Final test mAP50 / mAP50:95 |
| --- | --- | --- |
| YOLO11n | 0.6250 / 0.3383 | Not tested |
| **YOLO11s** | **0.6131 / 0.3510** | **0.6314 / 0.3270** |
| DETR-ResNet50 | 0.4377 / 0.1746 | Not tested |

YOLO11s test AP50 / AP50:95 by class: panel `0.7467 / 0.3158`, number `0.7934 / 0.5246`, route `0.4116 / 0.2027`, destination `0.5739 / 0.2651`. At the validation-selected confidence `0.50`, one false positive occurred across 12 zero-box test images; median / p95 inference was `26.46 / 36.57 ms`. Full training settings, per-class validation AP, hashes and evidence paths are in the [v4 results report](../Obj-Dect-OCR/object_detection/01760a841333/runs/v4-reviewed-01/RESULTS.md). The older result below uses a different dataset and is historical.

**Validation-only follow-up:** NMS IoU `0.50` scored mAP50:95 `0.3532` versus `0.3510` at the recorded `0.70` setting. The final test belongs to `0.70`; the later [NMS and crop review](../Obj-Dect-OCR/object_detection/01760a841333/runs/v4-reviewed-01/validation_review/REVIEW.md) is exploratory. The [v4 configuration](../Obj-Dect-OCR/object_detection/01760a841333/runs/v4-reviewed-01/detector_config.json) preserves the tested settings. Jy/Brendan human annotation review is on hold.

### Recorded `baseline-01` benchmark results

The following measurements were saved on 2 November 2026 using dataset manifest `bf938f526d2c01f1960b9af018f1fd1dc78a431e725037c84d1707dfcc5b826c`, CUDA, 640-pixel input, seed 42, and the best validation-selected checkpoint from each training run.

| Candidate | Train mAP50 / mAP50:95 | Valid mAP50 / mAP50:95 | Test mAP50 / mAP50:95 | Test median / p95 ms | Test negative FP |
|---|---|---|---|---|---:|
| YOLO11n | 0.8910 / 0.7451 | 0.5709 / 0.3466 | 0.7165 / 0.4139 | 40.21 / 123.25 | 0 |
| **YOLO11s** | **0.9140 / 0.8024** | **0.6734 / 0.4274** | 0.7077 / 0.3938 | **34.22** / 128.14 | 0 |
| DETR-ResNet50 | 0.3446 / 0.1504 | 0.3127 / 0.1359 | 0.2873 / 0.1251 | 83.79 / 191.24 | 7 |

Test per-class AP:

| Candidate | EDS panel | Number | Route / landmark | Destination |
|---|---|---|---|---|
| YOLO11n | 0.7088 / 0.3928 | 0.8356 / 0.5593 | 0.6677 / 0.3455 | 0.6539 / 0.3578 |
| **YOLO11s** | **0.8478 / 0.3865** | 0.7950 / **0.5704** | 0.6310 / **0.3476** | 0.5573 / 0.2706 |
| DETR-ResNet50 | 0.5560 / 0.2707 | 0.3446 / 0.1508 | 0.1168 / 0.0347 | 0.1317 / 0.0441 |

Each per-class entry is `mAP50 / mAP50:95`. YOLO11s remains the selected model because selection was based on the highest validation mAP50:95 (`0.4274`), before opening the held-out test. Its validation threshold sweep had the highest F1 at confidence `0.20` (`0.7032`) with no false positives on four reviewed validation negatives. NMS IoU remained `0.70`.

**Recorded execution deviation:** the final Run All used `BENCHMARK_MODELS=('yolo11n', 'yolo11s', 'detr-resnet-50')` and `OPERATING_CONF=0.30`, so the test report contains all three models and negative false-positive counts at `0.30`. The mAP calculation used predictions from score `0.001`, but the test did not measure the validation-selected operating threshold `0.20`. Do not use the test results to reselect a model or tune settings, and do not rerun the test to optimize them.

**Quantitative interpretation:** YOLO11s provides the strongest validation result and lowest median test latency. Number detection is its strongest localized class. DETR is slower, has substantially lower AP, and produced false positives on reviewed negatives. The 20-image test split is small, including only two reviewed negatives.

**Validation qualitative review:** the ten highest-priority YOLO11s cases were inspected at the validation-selected confidence `0.20`. The ranked cases contained 61 GT boxes: 52 occupied less than 1% of the image and 36 less than 0.25%. Diagnostic class-correct matching counted 109 TP, 47 FP and 45 FN across all 39 validation images, with 17 destination-to-route and 10 route-to-destination confusion candidates. Blur, glare, oblique perspective, crowding and partial occlusion also made several crops unsuitable for OCR. None of four reviewed validation negatives produced a prediction. These counts are diagnostics, not COCO mAP. The review package is under `Obj-Dect-OCR/object_detection/bf938f526d2c/runs/baseline-01/yolo11s/qualitative_validation/`.

**Task 3 annotation/split audit:** a human-review packet now covers the 137 original training, 39 validation and 20 test images. Structural heuristics flag 15 images containing 17 panels without associated content boxes, and the packet inventories all 22 original zero-box images. Twelve timestamp-derived filename groups cross splits, including same-minute screenshot/image sequences. This is a recording-level isolation warning, not proof of leakage; source/capture records are required to decide it. No labels or splits were changed, and the checklist remains open pending Brendan/Jy decisions. Evidence is under `Obj-Dect-OCR/object_detection/bf938f526d2c/runs/baseline-01/data_review/`.

**Application handoff status:** the selected YOLO11s checkpoint was installed and SHA-256 verified locally. The app now requests all four model classes and explicitly maps them to stable API labels: `front_eds→eds`, `front_eds_num→bus_number`, `front_eds_route→route`, and `front_eds_destination→destination`. Real CUDA verification detected two displays and all four field types in a multi-bus image (five-run detector median `35.249 ms`). A four-frame uploaded-video fixture verified decode, timestamps, association and detector flow (`73.706 ms` median / `145.758 ms` p95 total); a labelled still exercised the simulated camera response path. Production OCR/reference matching and a physical-camera field run remain open because local OCR assets, Tesseract, the route CSV and camera evidence are unavailable. Evidence is under `Obj-Dect-OCR/object_detection/bf938f526d2c/runs/baseline-01/deployment/`.

**EasyOCR integration status:** English EasyOCR assets were installed locally and the real CPU adapter processed YOLO11s number/route/destination crops. Across six diagnostic runs, OCR latency was `184.363 ms` median / `382.270 ms` p95 and total processing was `231.409 ms` median / `558.472 ms` p95. Raw readings on the small/distant cases were often wrong or below the `0.25` acceptance threshold, consistent with the qualitative crop review. No accuracy is claimed without reviewed transcriptions. Missing route-reference data now degrades to an unmatched raw-OCR result rather than disabling the app. Tesseract, fine-tuned TrOCR and a physical-camera moving-bus field run remain open.

In [ ]:
"""Format measured detector benchmarks as separate train, validation and test tables."""

from statistics import mean


SPLITS = (("train", "Train"), ("valid", "Validation"), ("test", "Test"))


def percentage(value):
    if value is None or value < 0:
        return "—"
    return f"{100 * value:.1f}"


def render_split_tables(benchmark, inventory, class_names, run_id):
    """Return Markdown; an unmeasured test split is explicitly left pending."""
    lines = [f"# Detector benchmark: {run_id}", ""]
    for split_key, title in SPLITS:
        lines.extend((f"## {title}", ""))
        measured = [(name, splits[split_key]) for name, splits in benchmark.items()
                    if split_key in splits]
        if not measured:
            lines.extend(("Held out; no measured results for this split yet.", ""))
            continue
        split_inventory = inventory.get(split_key, {})
        counts = split_inventory.get("boxes_by_class", {})
        lines.extend((
            f"Images: {split_inventory.get('images', '—')}; "
            f"negative images: {split_inventory.get('negative_images', '—')}.",
            "",
            "| Model | Class | Boxes (n) | mAP50 (%) | mAP50:95 (%) | Recall mAR@100 (%) |",
            "| --- | --- | ---: | ---: | ---: | ---: |",
        ))
        for model, metrics in measured:
            by_class = metrics.get("by_class", {})
            for class_name in class_names:
                class_metrics = by_class.get(class_name, {})
                lines.append(
                    f"| {model} | {class_name} | {counts.get(class_name, 0):,} | "
                    f"{percentage(class_metrics.get('map50'))} | "
                    f"{percentage(class_metrics.get('map50_95'))} | "
                    f"{percentage(class_metrics.get('mar_100'))} |"
                )
            recall = metrics.get("mar_100")
            if recall is None:
                measured_recall = [by_class[name]["mar_100"] for name in class_names
                                   if name in by_class and by_class[name].get("mar_100") is not None
                                   and by_class[name]["mar_100"] >= 0]
                recall = mean(measured_recall) if measured_recall else None
            lines.append(
                f"| **{model}** | **All (avg)** | **{sum(counts.values()):,}** | "
                f"**{percentage(metrics.get('map50'))}** | "
                f"**{percentage(metrics.get('map50_95'))}** | "
                f"**{percentage(recall)}** |"
            )
        lines.append("")
    lines.append("Values are measured on the named split. A dash means the class metric is unavailable.")
    lines.append("")
    return "\n".join(lines)


In [ ]:
if DATA_READY and BENCHMARK:
    import importlib.metadata as metadata
    report = {'dataset_root': str(DATASET_ROOT), 'dataset_manifest_sha256': DATA_FINGERPRINT,
              'inventory': inventory, 'class_map': {str(k): v for k, v in COCO_TO_INDEX.items()},
              'run_id': RUN_ID, 'models': MODEL_PATHS, 'training_history': TRAINING_HISTORY,
              'benchmark': BENCHMARK, 'validation_threshold_sweep': THRESHOLD_SWEEP,
              'settings': {'yolo_epochs': YOLO_EPOCHS,
              'detr_epochs': DETR_EPOCHS, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
              'seed': SEED, 'device': DEVICE, 'operating_conf': OPERATING_CONF,
              'yolo_nms_iou': YOLO_NMS_IOU, 'final_test_run': RUN_FINAL_TEST},
              'environment': {'platform': platform.platform(), 'python': sys.version,
              'torch': metadata.version('torch'), 'ultralytics': metadata.version('ultralytics'),
              'transformers': metadata.version('transformers'), 'torchmetrics': metadata.version('torchmetrics')}}
    suffix = '_'.join(BENCHMARK_MODELS)
    phase = 'test' if RUN_FINAL_TEST else 'validation'
    destination = OUTPUT_ROOT / DATA_FINGERPRINT[:12] / 'runs' / RUN_ID / f'benchmark_{suffix}_{phase}.json'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(report, indent=2), encoding='utf-8')
    for name in BENCHMARK_MODELS:
        archive_benchmark(ROOT, DATA_FINGERPRINT, name, phase, report, RUN_ID)
    from IPython.display import Markdown, display
    tables = render_split_tables(BENCHMARK, inventory, CLASS_NAMES, RUN_ID)
    table_path = destination.with_name(destination.stem + '_tables.md')
    table_path.write_text(tables, encoding='utf-8')
    display(Markdown(tables))
    print('Evidence saved:', destination)
    print('Split tables saved:', table_path)
else:
    print('No measured benchmark report was written. The table remains pending.')

## 6. YOLO loss research and how to apply it

The supplied slides illustrate three YOLO detection-loss parts. The pinned Ultralytics YOLO11 implementation uses **CIoU box loss**, **BCE classification loss**, and **distribution focal loss (DFL)** for the four box-edge distance distributions. In shorthand, `L = box_gain × L_CIoU + cls_gain × L_BCE + dfl_gain × L_DFL`; the trainer logs `train/box_loss`, `train/cls_loss`, `train/dfl_loss` and corresponding validation losses. CIoU accounts for overlap, centre distance and aspect ratio. DFL is a box-distance regression term, *not* the same as classification focal loss. The slide's BCE-to-focal arrow is an optional experiment, not the default YOLO11 loss. DETR has a different matching/classification/box objective, so do not compare raw loss values across architectures.

**Project application:** keep YOLO11 defaults for the controlled baseline and record all three curves. Inspect per-class AP and negative-image false positives first. If small EDS or route boxes remain poorly localized, change one documented parameter at a time (`box` or `dfl`) and compare validation mAP50:95 and crop quality. If a verified class imbalance causes route/destination mistakes, test `cls` gain and sampling before considering a separately implemented focal-loss ablation. Do not claim focal loss is active merely because DFL appears in the logs. Retain the same test split and class mapping across ablations.

Primary references: [Ultralytics detection loss source](https://docs.ultralytics.com/reference/utils/loss), [Ultralytics training settings](https://docs.ultralytics.com/modes/train), [CIoU paper](https://arxiv.org/abs/1911.08287), [Focal Loss paper](https://arxiv.org/abs/1708.02002), [Generalized Focal Loss paper](https://arxiv.org/abs/2006.04388), [DETR paper](https://arxiv.org/abs/2005.12872).
